# AdaBoost 실습

In [1]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve
import numpy as np
import matplotlib.pyplot as plt

In [2]:
X, y = load_breast_cancer().data, load_breast_cancer().target

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
ab = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # max_depth=1, 약한분류기
    n_estimators=50, # defalut 값
    learning_rate=1, # defalut 값
    algorithm='SAMME',
    random_state=42
)

In [5]:
ab.fit(X_train, y_train)

/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


,estimator,DecisionTreeC...r(max_depth=1)
,n_estimators,50
,learning_rate,1
,algorithm,'SAMME'
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [6]:
ab.score(X_test, y_test)

0.9649122807017544

In [7]:
y_pred = ab.predict(X_test)

In [8]:
confusion_matrix(y_test, y_pred)

array([[40,  3],
       [ 1, 70]])

In [9]:
print(classification_report(y_test, y_pred, target_names=load_breast_cancer().target_names))

              precision    recall  f1-score   support

   malignant       0.98      0.93      0.95        43
      benign       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [10]:
y_proba = ab.predict_proba(X_test)

In [11]:
roc_auc_score(y_test, y_proba[:,1])

0.9924664264657713

# AdaBoost 튜닝

In [25]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# 파라미터 그리드
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.5, 1.0, 1.5],
    'estimator': [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2)
    ]
}

# Grid Search
ada_grid = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

ada_grid.fit(X_train, y_train)
print("최적 파라미터:", ada_grid.best_params_)
print(f"최적 CV 점수: {ada_grid.best_score_:.4f}")
print(f"테스트 정확도: {ada_grid.score(X_test, y_test):.4f}")

최적 파라미터: {'estimator': DecisionTreeClassifier(max_depth=1), 'learning_rate': 1.0, 'n_estimators': 50}
최적 CV 점수: 0.9802
테스트 정확도: 0.9649


# XGBoost - Sklearn

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
model.score(X_test, y_test)

0.956140350877193

# XGBoost - DMatrix

In [20]:
# DMatrix 생성 (XGBoost 전용 데이터 구조 - 메모리 효율적)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, y_test)

In [19]:
# 파라미터 설정
params = {
    'objective': 'binary:logistic',
    'max_depth': 3,
    'learning_rate': 0.1,
    'eval_metric': 'logloss'
}

# 학습 (조기 종료 포함)
model = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=10,
    verbose_eval=20
)

[0]	train-logloss:0.58035	test-logloss:0.58723
[20]	train-logloss:0.10597	test-logloss:0.15585
[40]	train-logloss:0.04004	test-logloss:0.11754
[60]	train-logloss:0.02172	test-logloss:0.10976
[68]	train-logloss:0.01817	test-logloss:0.10989


In [23]:
# 예측
y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)
print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")

정확도: 0.9561
